In [ ]:
import os
import pandas as pd
import numpy as np
import ast
import pingouin as pg
from sklearn.metrics import cohen_kappa_score

def analyze_with_pingouin(root_path, hospitals, filename='Ensemble_Weighted_Result.csv'):
    dfs = []
    for hop in hospitals:
        file_path = os.path.join(root_path, hop, filename)
        if not os.path.exists(file_path): continue
            
        df = pd.read_csv(file_path)
        
        # --- 核心转换步骤 ---
        def extract_prob(prob_str):
            try:
                # 将字符串 '[0.87, 0.12]' 转为列表 [0.87, 0.12]
                p_list = ast.literal_eval(prob_str)
                # 提取类 1 的概率（假设 index 1 是目标类）
                return p_list[1] 
            except:
                return np.nan

        df['base_id'] = df['slide_id'].apply(lambda x: str(x).split('-')[0])
        # 执行转换
        df[f'probs_{hop}'] = df['probs'].apply(extract_prob)
        df = df.rename(columns={'prediction': f'pred_{hop}'})
        
        dfs.append(df[['base_id', f'probs_{hop}', f'pred_{hop}', 'label']])

    # 合并数据
    final_df = dfs[0]
    for next_df in dfs[1:]:
        final_df = pd.merge(final_df, next_df, on=['base_id', 'label'], how='inner')

    # 移除任何包含空值的行（转换失败的情况）
    final_df = final_df.dropna()

    # --- 1. 指标波动分析 (Standard Deviation) ---
    prob_cols = [f'probs_{hop}' for hop in hospitals]
    final_df['prob_std'] = final_df[prob_cols].std(axis=1)
    
    print("-" * 30)
    print(f"## 1. 指标波动分析 (SD)")
    print(f"分析样本数: {len(final_df)}")
    print(f"平均概率标准差 (Mean SD): {final_df['prob_std'].mean():.4f}")
    print(f"SD 越小，说明模型对不同染色中心越不敏感（更稳定）。")

    # --- 2. 一致性分析 (ICC) ---
    # 将宽表转长表
    long_df = final_df.melt(id_vars=['base_id'], value_vars=prob_cols, 
                            var_name='hospital', value_name='prob_score')
    
    # 计算 ICC
    icc = pg.intraclass_corr(data=long_df, targets='base_id', raters='hospital', ratings='prob_score')
    
    print("\n## 2. 一致性分析 (ICC)")
    # ICC3k: 固定评分者（6家医院）的平均一致性
    target_row = icc[icc['Type'].str.contains(r'\(C,\s*k\)', regex=True)]

    if not target_row.empty:
        val = target_row['ICC'].values[0]
        ci = target_row['CI95'].values[0]
        print(f"多中心一致性 (ICC C,k): {val:.4f}")
        print(f"95% 置信区间: {ci}")
    else:
        # 如果没找到 k 类型，先打印全表查看 Type 列的具体写法
        print("未匹配到特定 ICC 类型，全表结果如下：")
        print(icc[['Type', 'ICC', 'CI95']])

    # --- 3. 分类一致性 (Kappa) ---
    pred_cols = [f'pred_{hop}' for hop in hospitals]
    kappas = []
    for i in range(len(hospitals)):
        for j in range(i + 1, len(hospitals)):
            k = cohen_kappa_score(final_df[pred_cols[i]], final_df[pred_cols[j]])
            kappas.append(k)
    
    print(f"\n## 3. 分类一致性 (Kappa)")
    print(f"平均 Cohen's Kappa: {np.mean(kappas):.4f}")
    print("-" * 30)

    return final_df

# 运行分析
hospitals = [d for d in os.listdir('Consistent') if os.path.isdir(os.path.join('Consistent', d))]
final_results = analyze_with_pingouin('Consistent', hospitals)